# LangGraph Chatbot with Hybrid Retrieval and Streaming

This tutorial builds a production-ready RAG chatbot using:

- **LangGraph** — stateful, multi-step agent graph
- **LangChain + OpenAI** — LLM and embedding models
- **Hybrid Retrieval** — BM25 (keyword) + FAISS (semantic) combined via `EnsembleRetriever`
- **Streaming** — token-level streaming for real-time responses

```
User Query
    │
    ▼
┌──────────┐     ┌──────────────────────────────┐
│  START   │────▶│  retrieve                    │
└──────────┘     │  BM25 + Embeddings → context │
                 └──────────────┬───────────────┘
                                │
                                ▼
                 ┌──────────────────────────────┐
                 │  generate                    │
                 │  LLM (streaming) → answer    │
                 └──────────────┬───────────────┘
                                │
                                ▼
                             END
```

## 1. Installation

In [ ]:
%pip install -qU \
    langgraph \
    langchain \
    langchain-openai \
    langchain-community \
    faiss-cpu \
    rank_bm25 \
    python-dotenv \
    tiktoken

## 2. Setup — API Keys and Models

Create a `.env` file in the project root with:
```
OPENAI_API_KEY=sk-...
```

In [ ]:
from dotenv import load_dotenv, find_dotenv
import os

load_dotenv(find_dotenv())

# Verify key is loaded
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not found — add it to your .env file"

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# streaming=True enables token-level streaming via .stream() / astream_events()
llm = ChatOpenAI(model="gpt-4o", temperature=0, streaming=True)

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

print("LLM:", llm.model_name)
print("Embeddings model: text-embedding-3-small")

## 3. Knowledge Base — Sample Documents

We create a corpus of documents about LangChain, LangGraph, and RAG.  
In production you would load PDFs, web pages, or a database here.

In [ ]:
from langchain_core.documents import Document

raw_texts = [
    # LangChain
    """LangChain is an open-source framework for building applications powered by large language models (LLMs). 
    It provides composable building blocks — prompts, chains, agents, tools, memory, and retrievers — 
    that developers can assemble to create complex AI applications. LangChain supports integrations with 
    over 60 LLM providers including OpenAI, Anthropic, and Cohere.""",

    """LangChain Expression Language (LCEL) is the recommended way to compose LangChain components. 
    It uses the pipe operator (|) to chain Runnables together: prompt | llm | output_parser. 
    LCEL supports streaming, async, batch, and parallel execution out of the box, 
    and provides an elegant way to build both simple chains and complex agent workflows.""",

    # LangGraph
    """LangGraph is a library for building stateful, multi-step agentic applications using a graph abstraction. 
    Unlike a simple chain, LangGraph represents your application as a directed graph where nodes are 
    functions (or LLM calls) and edges define the flow of data. It supports cycles, branching, 
    human-in-the-loop, and persistent state — making it ideal for complex agent architectures.""",

    """LangGraph StateGraph is the core primitive in LangGraph. You define a TypedDict as the graph state, 
    add nodes (Python functions), connect them with edges, and compile the graph. The state is passed 
    between nodes and accumulated using reducer functions. The special `add_messages` reducer appends 
    messages to the conversation history without overwriting them.""",

    """LangGraph supports three streaming modes: 'values' streams the full graph state after each step, 
    'updates' streams only the node output deltas, and 'messages' streams individual LLM token chunks 
    for real-time token-level output. You can combine modes by passing a list: stream_mode=['updates', 'messages'].""",

    # RAG
    """Retrieval-Augmented Generation (RAG) is a technique that grounds LLM responses in external knowledge. 
    Instead of relying solely on the model's parametric memory, RAG retrieves relevant documents at 
    query time and injects them into the prompt as context. This reduces hallucinations and lets the 
    model answer questions about private or up-to-date information.""",

    """Dense retrieval uses neural embedding models to encode documents and queries into vector space. 
    Semantic similarity is measured with cosine similarity or dot product. FAISS (Facebook AI Similarity Search) 
    is a popular library for efficient approximate nearest-neighbor search over millions of embeddings. 
    OpenAI's text-embedding-3-small model produces 1536-dimensional vectors at low cost.""",

    """BM25 (Best Match 25) is a classical sparse retrieval algorithm based on TF-IDF with document 
    length normalization. It excels at exact keyword matching and handles out-of-vocabulary terms 
    that embedding models might miss. BM25 is fast, requires no GPU, and needs no training data — 
    making it a strong baseline for any retrieval system.""",

    # Hybrid Retrieval
    """Hybrid retrieval combines sparse retrieval (BM25) and dense retrieval (embeddings) to get the 
    best of both worlds. BM25 handles exact keyword matches well; dense retrieval captures semantic 
    similarity. The EnsembleRetriever in LangChain merges results using Reciprocal Rank Fusion (RRF), 
    a rank-aggregation method that is robust to score distribution differences between retrievers.""",

    """Reciprocal Rank Fusion (RRF) scores each document as the sum of 1/(k + rank_i) across retrievers, 
    where k=60 by default. Documents appearing at the top of multiple retriever result lists receive 
    the highest combined score. RRF is parameter-light and outperforms linear score interpolation 
    in most benchmarks without requiring score normalization.""",

    # FAISS
    """FAISS from_documents() takes a list of LangChain Documents and an embedding model, embeds all 
    document texts, and stores them in an index. The as_retriever() method wraps it in a LangChain 
    retriever interface. You can persist the index to disk with save_local() and reload with 
    load_local() to avoid re-embedding on every run.""",

    # OpenAI models
    """GPT-4o is OpenAI's flagship multimodal model as of 2024. It accepts text and image inputs and 
    returns text output. GPT-4o is faster and cheaper than GPT-4 Turbo while matching or exceeding 
    its performance on most benchmarks. It supports a 128k token context window and function calling 
    with parallel tool use.""",

    """OpenAI's streaming API returns Server-Sent Events (SSE) with delta chunks as the model generates 
    tokens. In LangChain, setting streaming=True on ChatOpenAI and calling .stream() yields 
    AIMessageChunk objects one token at a time. This enables responsive UIs that show text appearing 
    incrementally rather than waiting for the full response.""",

    # Agents
    """A LangGraph agent typically alternates between an LLM call (which may decide to use tools) 
    and a tool execution step. The graph loops until the LLM produces a response without requesting 
    a tool call. This ReAct-style loop (Reasoning + Acting) is the foundation of most LangGraph agent 
    implementations and can be extended with memory, human approval steps, and custom tools.""",

    """LangGraph checkpointers enable persistent conversation memory across sessions. When you compile 
    a graph with a checkpointer (e.g., MemorySaver or PostgresSaver), the graph state is saved after 
    each node execution. Passing a thread_id in the config lets you resume a conversation exactly 
    where it left off, even across process restarts.""",
]

documents = [Document(page_content=text.strip()) for text in raw_texts]

print(f"Loaded {len(documents)} documents")
print(f"\nExample document:\n{documents[0].page_content[:200]}...")

## 4. Hybrid Retriever: BM25 + FAISS

We build two independent retrievers and combine them with `EnsembleRetriever`.

| Retriever | Type | Strength |
|-----------|------|----------|
| BM25 | Sparse (keyword) | Exact matches, rare terms |
| FAISS | Dense (semantic) | Synonyms, paraphrases |
| **Ensemble** | **Hybrid** | **Both** |

Results are merged with **Reciprocal Rank Fusion (RRF)** — a rank-aggregation method that rewards documents appearing high in multiple retriever lists.

In [ ]:
from langchain_community.retrievers import BM25Retriever

# BM25 — sparse keyword retriever (no embeddings needed)
bm25_retriever = BM25Retriever.from_documents(documents)
bm25_retriever.k = 4  # return top-4 documents

# Quick test
bm25_results = bm25_retriever.invoke("BM25 keyword retrieval")
print(f"BM25 returned {len(bm25_results)} docs")
print(f"Top result: {bm25_results[0].page_content[:100]}...")

In [ ]:
from langchain_community.vectorstores import FAISS

# Dense retriever — embeds all documents (API call to OpenAI)
print("Building FAISS index (embedding documents)...")
vectorstore = FAISS.from_documents(documents, embeddings)
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

# Quick test
dense_results = dense_retriever.invoke("semantic similarity vector search")
print(f"FAISS returned {len(dense_results)} docs")
print(f"Top result: {dense_results[0].page_content[:100]}...")

In [ ]:
from langchain.retrievers import EnsembleRetriever

# Weights control the blend: 0.4 BM25 + 0.6 dense
# RRF is applied *after* weighting to merge the ranked lists
hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, dense_retriever],
    weights=[0.4, 0.6],
)

# Test hybrid retrieval
query = "how does hybrid retrieval combine BM25 and embeddings?"
hybrid_results = hybrid_retriever.invoke(query)

print(f"Hybrid retriever returned {len(hybrid_results)} docs\n")
for i, doc in enumerate(hybrid_results, 1):
    print(f"--- Doc {i} ---")
    print(doc.page_content[:150], "...\n")

## 5. LangGraph State and Nodes

The graph state holds two things:
- `messages` — the full conversation history (uses `add_messages` reducer to accumulate)
- `context` — the retrieved document text injected into the prompt

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage
from langgraph.graph.message import add_messages


class ChatState(TypedDict):
    # add_messages reducer: new messages are appended, not overwritten
    messages: Annotated[list[BaseMessage], add_messages]
    # retrieved context injected into the system prompt
    context: str

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder


# ── Node 1: Retrieve ──────────────────────────────────────────────────────────
def retrieve(state: ChatState) -> dict:
    """Runs the hybrid retriever on the latest user message."""
    last_human = next(
        (m for m in reversed(state["messages"]) if isinstance(m, HumanMessage)),
        None,
    )
    query = last_human.content if last_human else ""
    docs = hybrid_retriever.invoke(query)
    context = "\n\n".join(
        f"[Document {i+1}]\n{doc.page_content}" for i, doc in enumerate(docs)
    )
    return {"context": context}


# ── Node 2: Generate ──────────────────────────────────────────────────────────
SYSTEM_PROMPT = """You are a knowledgeable AI assistant specializing in LangChain, LangGraph, and RAG systems.
Use the retrieved context below to answer the user's question accurately.
If the context does not contain enough information, say so and answer from your own knowledge.

Retrieved context:
{context}"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    MessagesPlaceholder(variable_name="messages"),
])

generate_chain = prompt | llm


def generate(state: ChatState) -> dict:
    """Calls the LLM with conversation history and retrieved context."""
    response = generate_chain.invoke({
        "context": state["context"],
        "messages": state["messages"],
    })
    return {"messages": [response]}


print("Nodes defined: retrieve, generate")

## 6. Build and Compile the Graph

The graph has a simple linear topology:

```
START → retrieve → generate → END
```

We use `MemorySaver` as a checkpointer so the conversation history persists across turns within the same `thread_id`.

In [ ]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

# Build the graph
builder = StateGraph(ChatState)

builder.add_node("retrieve", retrieve)
builder.add_node("generate", generate)

builder.add_edge(START, "retrieve")
builder.add_edge("retrieve", "generate")
builder.add_edge("generate", END)

# MemorySaver persists state in-memory per thread_id
memory = MemorySaver()
chatbot = builder.compile(checkpointer=memory)

print("Graph compiled successfully")

In [ ]:
# Visualize the graph topology
from IPython.display import Image, display

try:
    display(Image(chatbot.get_graph().draw_mermaid_png()))
except Exception:
    # Fallback: print mermaid source if graphviz is not available
    print(chatbot.get_graph().draw_mermaid())

## 7. Basic Chat (No Streaming)

Use `chatbot.invoke()` for a single blocking call.  
The `thread_id` in the config links turns into a single conversation.

In [ ]:
def chat(question: str, thread_id: str = "default") -> str:
    """Send a message and return the assistant response."""
    config = {"configurable": {"thread_id": thread_id}}
    initial_state = {
        "messages": [HumanMessage(content=question)],
        "context": "",
    }
    result = chatbot.invoke(initial_state, config=config)
    return result["messages"][-1].content


# First turn
answer = chat("What is LangGraph and how does it differ from a simple LangChain chain?", thread_id="demo-1")
print(answer)

In [ ]:
# Second turn — the chatbot remembers the previous exchange (same thread_id)
answer = chat("And what streaming modes does it support?", thread_id="demo-1")
print(answer)

## 8. Streaming — Token by Token Output

LangGraph's `stream_mode="messages"` yields `(chunk, metadata)` tuples.  
When the chunk comes from the `generate` node and is an `AIMessageChunk`, it contains a token fragment.

In [ ]:
from langchain_core.messages import AIMessageChunk


def chat_stream(question: str, thread_id: str = "stream-default") -> str:
    """Stream the response token by token and return the full answer."""
    config = {"configurable": {"thread_id": thread_id}}
    initial_state = {
        "messages": [HumanMessage(content=question)],
        "context": "",
    }

    print(f"User: {question}")
    print("Assistant: ", end="", flush=True)

    full_response = ""
    for chunk, metadata in chatbot.stream(
        initial_state,
        config=config,
        stream_mode="messages",
    ):
        # Only print tokens generated by the 'generate' node
        if (
            isinstance(chunk, AIMessageChunk)
            and metadata.get("langgraph_node") == "generate"
            and chunk.content
        ):
            print(chunk.content, end="", flush=True)
            full_response += chunk.content

    print()  # newline after streaming completes
    return full_response


_ = chat_stream(
    "Explain how BM25 works and why we combine it with embedding-based retrieval.",
    thread_id="stream-1",
)

In [ ]:
_ = chat_stream(
    "What is Reciprocal Rank Fusion and how does EnsembleRetriever use it?",
    thread_id="stream-2",
)

## 9. Async Streaming with `astream_events`

For production applications (FastAPI, async servers) use the async streaming API which provides fine-grained control via event types.

In [ ]:
import asyncio


async def chat_async_stream(question: str, thread_id: str = "async-default") -> str:
    """Async token streaming using astream_events (v2 API)."""
    config = {"configurable": {"thread_id": thread_id}}
    initial_state = {
        "messages": [HumanMessage(content=question)],
        "context": "",
    }

    print(f"User: {question}")
    print("Assistant: ", end="", flush=True)

    full_response = ""
    async for event in chatbot.astream_events(initial_state, config=config, version="v2"):
        event_type = event["event"]
        node = event.get("metadata", {}).get("langgraph_node", "")

        # 'on_chat_model_stream' fires for each token from the LLM
        if event_type == "on_chat_model_stream" and node == "generate":
            chunk = event["data"]["chunk"]
            if chunk.content:
                print(chunk.content, end="", flush=True)
                full_response += chunk.content

    print()
    return full_response


# Run the async function in a Jupyter-compatible way
await chat_async_stream(
    "How does LangGraph's MemorySaver enable multi-turn conversations?",
    thread_id="async-1",
)

## 10. Multi-Turn Conversation Demo

Because we compiled with `MemorySaver`, each call with the same `thread_id` resumes from where the previous turn ended. The full message history is stored in the checkpointer, not in a Python variable — so it survives page reloads and restarts (within the same process).

In [ ]:
THREAD = "tutorial-session"
DIVIDER = "-" * 60

questions = [
    "What is RAG and why is it useful?",
    "How does the retrieval step work in a RAG pipeline?",
    "What are the advantages of adding BM25 to semantic search?",
    "Can you summarise what we discussed so far in three bullet points?",
]

for q in questions:
    print(DIVIDER)
    response = chat_stream(q, thread_id=THREAD)
    print()

## 11. Inspect Conversation State

With a checkpointer you can inspect (and even edit) the stored state for any thread at any time.

In [ ]:
# Retrieve the current state for our tutorial session
config = {"configurable": {"thread_id": THREAD}}
state_snapshot = chatbot.get_state(config)

messages = state_snapshot.values["messages"]
print(f"Messages in conversation history: {len(messages)}\n")

for msg in messages:
    role = "User" if isinstance(msg, HumanMessage) else "Assistant"
    print(f"{role}: {msg.content[:120]}..." if len(msg.content) > 120 else f"{role}: {msg.content}")
    print()

## 12. Retrieval Quality Check

Compare what BM25, dense retrieval, and hybrid retrieval each surface for the same query.

In [ ]:
def compare_retrievers(query: str, k: int = 3):
    """Show top-k results from each retriever side by side."""
    bm25_retriever.k = k
    dense_r = vectorstore.as_retriever(search_kwargs={"k": k})
    hybrid_r = EnsembleRetriever(
        retrievers=[bm25_retriever, dense_r], weights=[0.4, 0.6]
    )

    print(f"Query: '{query}'\n")
    print("=" * 60)

    for label, retriever in [("BM25", bm25_retriever), ("Dense (FAISS)", dense_r), ("Hybrid", hybrid_r)]:
        results = retriever.invoke(query)
        print(f"\n{'─'*20} {label} {'─'*20}")
        for i, doc in enumerate(results, 1):
            snippet = doc.page_content[:120].replace("\n", " ")
            print(f"  {i}. {snippet}...")


compare_retrievers("LLM streaming tokens real-time")

In [ ]:
compare_retrievers("BM25 keyword exact match")

## Summary

We built a complete RAG chatbot with the following components:

| Component | Implementation | Why |
|-----------|---------------|-----|
| LLM | `ChatOpenAI(gpt-4o, streaming=True)` | Latest model, token streaming support |
| Embeddings | `OpenAIEmbeddings(text-embedding-3-small)` | Fast & affordable dense vectors |
| Sparse retriever | `BM25Retriever` | Keyword matching, no GPU needed |
| Dense retriever | `FAISS.as_retriever()` | Semantic similarity |
| Hybrid retriever | `EnsembleRetriever` (RRF, weights 0.4/0.6) | Best of both worlds |
| Graph framework | `LangGraph StateGraph` | Stateful multi-step flow |
| Memory | `MemorySaver` checkpointer | Persistent multi-turn history |
| Streaming | `stream_mode="messages"` / `astream_events` | Real-time token output |

### Next Steps

- **Routing**: Add a `should_retrieve` conditional edge to skip retrieval for greetings / small-talk
- **Query rewriting**: Add a node that rephrases the user query before retrieval
- **Re-ranking**: Add a cross-encoder re-ranker node between retrieve and generate
- **Persistence**: Replace `MemorySaver` with `PostgresSaver` for production persistence
- **Streaming server**: Expose `astream_events` via a FastAPI `StreamingResponse`